# 准备 CC3M 数据集和 DataLoader

在 [基于 transformers 搭建 Llava.ipynb](<基于 transformers 搭建 Llava.ipynb>) 中，我们搭建好自己的 LLaVA 模型并保存到 `show_model/model001`。目前我们的模型只有 CLIP-ViT 和 Qwen-1.5-4B 各自的预训练权重，LLaVA 模型的对齐单元（MLP）以及对 LLM 做的训练还没有进行，所以模型还并不具备读图能力。

为了训练出一个与 LLaVA 功能相似的模型，我们选择用 [LLaVA-CC3M-Pretrain-595K](https://huggingface.co/datasets/liuhaotian/LLaVA-CC3M-Pretrain-595K) 的随机抽取的一个小子集，做**小规模**的 LLaVA 第一阶段（特征对齐）复现。

训练方式：

- 视觉塔（CLIP ViT-L/14-336）冻结；
- LLM（Qwen1.5-4B-Chat）用 LoRA 训练；
- 视觉-文本投影层（`multi_modal_projector`，两层 MLP）全量训练。

In [ ]:
# pip install transformers==4.41.2 peft==0.11.1 huggingface_hub modelscope

## 与原始 LLaVA 论文的差异

### LLaVA 训练过程及数据简介

在 [CLIP 和 Llama.md](<CLIP 和 Llama.md>) 中，提到 LLaVA 的原始训练过程分为两个阶段：

- **特征对齐预训练**：冻结 ViT 和 LLM 权重，使用**图像-文本对数据集**训练投影/对齐单元。  
- **端到端微调**：冻结 ViT 权重，使用**视觉指令数据集**同时调整对齐单元和 LLM 的权重。

具体而言，图像-文本对数据集就是我们将要使用的经过 LLaVA 团队筛选的 CC3M 数据集，视觉指令数据集则是 LLaVA 团队在 COCO 2017 数据集的基础上自行构建的。**前一个数据集中的指令语义非常单一，基本停留在图像内容描述层面**（如“这张图片里有什么？”）。后一个数据集才涉及图片具体的提问，甚至基于图片进行推理。

### 我们将要进行的训练过程

我们的训练过程只有一个阶段：

- **同时进行特征对齐 + LLM 的轻量视觉适配**：冻结 ViT 权重，使用**图像-文本对数据集**训练对齐单元和 LLM，其中 LLM 使用 LoRA 进行高效微调。

这种设计会带来一些需要留意的影响：

- **训练初期可能不如原版稳定**：Connector 和 LLM LoRA 同时从早期开始联合优化，缺少了原版 Stage 1 中 LLM 冻结提供的“安全环境”，损失曲线可能更易波动。
- **指令遵循能力有限**：由于训练数据是图文对而非指令数据，模型学到的主要是“用语言描述视觉特征”，而非原版 Stage 2 后的复杂指令跟随、多轮对话或基于图像的推理。它具备的是**基础的看图说话能力**，而不是完整的视觉指令遵循。
- **LoRA 的正则化效应提供了补偿**：相比全量微调 LLM，LoRA 只更新少量参数，降低了模型在缺乏独立对齐阶段时“跑偏”或遗忘预训练知识的风险，使这种单阶段联合训练在实践中相对可控。
- **性价比优势明显**：训练成本显著低于完整两阶段流程，适合在有限算力下快速得到一个基础可用的模型。

本仓库的目标是**在有限算力预算下，快速得到一个效果可接受、具备基本视觉指令跟随能力的模型**。完全模仿原论文进行训练成本较高，因此选择了这样的训练方式作为**更具性价比的起点**。如果后续需要更完整的指令遵循能力，可以在此基础上引入 LLaVA-Instruct-150K 和 COCO 2017 图片，补充第二阶段的指令微调。

## 下载数据集

魔塔社区并没有提供 LLaVA 的 CC3M 数据集。我们需要使用镜像站或 HuggingFace 主站进行数据集下载。这里默认在 AutoDL 上使用学术加速，从 HuggingFace 主站 下载数据集。以下命令需要安装好 `huggingface_hub`。

```bash
unset HF_ENDPOINT
source /etc/network_turbo
hf download liuhaotian/LLaVA-CC3M-Pretrain-595K --local-dir /root/autodl-tmp/CC3M
```

数据集默认下载到数据盘中，大小约 7GB。图片文件是在压缩包中的，后续需要根据需要解压。

### 数据格式

`chat.json` 的每条记录形如：

```json
{
  "conversations": [
    {"from": "human", "value": "Provide a brief description of the given image.\n<image>"},
    {"from": "gpt", "value": "olive oil is a healthy ingredient used liberally ."}
  ],
  "image": "GCC_train_002582585.jpg",
  "id": "GCC_train_002582585"
}
```

- `<image>` 已经包含在 human 的文本里，位置可能在开头或结尾；
- 图片名是扁平的 `GCC_train_*.jpg`，都在 `images.zip` 的根目录；
- 我们只需要把选中的图片从 zip 里取出来，不需要解压全部 59 万张。

### 检查数据

运行下方的代码块，对下载好的数据做数量上和读取功能上的检查。为了后续校验数据加载器能否被我们的模型使用，我们这里同时检查是否运行过 [基于 transformers 搭建 Llava.ipynb](<基于 transformers 搭建 Llava.ipynb>) 将初始 LLaVA 模型保存到指定路径。

In [ ]:
from pathlib import Path

MODEL_DIR = "show_model/model001"   # 搭建 notebook 保存的模型目录
DATA_ROOT = Path("/root/autodl-tmp/CC3M")   # 数据集下载路径

CHAT_PATH = DATA_ROOT / "chat.json"
IMAGE_ZIP = DATA_ROOT / "images.zip"

SUBSET_SIZE = 5000   # 训练样本数
VAL_SIZE = 100       # 验证样本数
SEED = 42

# 解压出来的图片的保存路径
IMAGE_DIR = DATA_ROOT / "images"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)

assert Path(MODEL_DIR).exists(), f"{MODEL_DIR} 不存在，请先运行搭建 notebook"
assert CHAT_PATH.exists(), f"{CHAT_PATH} 不存在"
assert IMAGE_ZIP.exists(), f"{IMAGE_ZIP} 不存在"

print("data root:", DATA_ROOT)
print("model dir:", Path(MODEL_DIR).resolve())

## 划分和解压

### 将数据集划分为训练集和验证集

在 `chat.json` 中随机抽出上面代码块指定的样本数量的样本（这里还没有加载图片，只是把图片对应的文本信息加载进来，后续直接通过 json 中保存的文件名解压对应的图片），加载进内存。

In [ ]:
import json
import random

chat_path = CHAT_PATH
print(chat_path)

# 把所有样本的对话部分和相关信息的 json 读入内存
with open(chat_path, encoding="utf-8") as f:
    all_records = json.load(f)

print("total records:", len(all_records))

# 随机取样本数中的 indices，通过 indices 取子集
rng = random.Random(SEED)
indices = rng.sample(range(len(all_records)), SUBSET_SIZE + VAL_SIZE)
subset = [all_records[i] for i in indices]
del all_records

# 保证每条样本的 human 文本里有且只有一个 <image>
for rec in subset:
    human = rec["conversations"][0]["value"]
    # 如果数量不对，默认头部添加 "<image>\n" 与删掉所有 <image> 的去掉头尾空白的原文本拼接
    if human.count("<image>") != 1:
        rec["conversations"][0]["value"] = "<image>\n" + human.replace("<image>", "").strip()

# subset 进一步划分为训练集和测试集
train_records = subset[:SUBSET_SIZE]
val_records = subset[SUBSET_SIZE:]

# 保存训练集和测试集 json 文件
with open(DATA_ROOT / "chat_train.json", "w", encoding="utf-8") as f:
    json.dump(train_records, f, ensure_ascii=False) # 非 ASCII 字符原样输出
with open(DATA_ROOT / "chat_val.json", "w", encoding="utf-8") as f:
    json.dump(val_records, f, ensure_ascii=False)

print("train:", len(train_records), "val:", len(val_records))

这里将 json 文件写入磁盘，后续使用时直接结合 json 和下面的解压路径中的对应 image 中读取数据样本。

### 解压图片

`images.zip` 里是 59 万多张扁平的 `GCC_train_*.jpg`；这里只解压选中的 5100 张到 `images/`，避免生成 59 万个文件。同时做 missing 统计，但通常用不上。如果进 missing 说明下载下来的 zip 中的图片和 json 中的匹配不上，**建议重新下载数据集**。

In [ ]:
import zipfile
from pathlib import Path
# 使用 python 进度条
from tqdm.auto import tqdm

selected_names = [rec["image"] for rec in subset]   # 根据随机抽样的结果，获取图片名字列表
todo = [name for name in selected_names if not (IMAGE_DIR / name).exists()] # 扫描解压后的图片路径，过滤出本地还不存在的图片
print("images to extract:", len(todo))  # 统计还需解压的图片数量

with zipfile.ZipFile(IMAGE_ZIP) as z:
    # zip 内部是扁平结构，这里保险起见，依旧使用遍历 zip 中所有文件的方式
    available = {
        Path(info.filename).name: info.filename     # Path(info.filename).name 取文件名部分（basename）
        for info in z.infolist()                    # z.infolist() 返回 zip 内所有条目（文件 + 目录）的元信息
        if not info.is_dir()                        # if not info.is_dir() 跳过目录条目，只保留文件
    }
    # available 是一个字典。这里是 图片名字符串：图片在 zip 中的路径
    # 因为这个数据集的 zip 里文件是扁平的，所以键与值应该是一样的。
    
    missing = [name for name in todo if name not in available]  # 找出 todo 里在 zip 中找不到的图片名
    print("missing in zip:", len(missing), missing[:3])

    for name in tqdm(todo):
        if name not in available:
            continue
        (IMAGE_DIR / name).write_bytes(z.read(available[name]))
        # z.read(available[name]) 从 zip 里读出该文件的完整二进制内容，返回 bytes
        # (IMAGE_DIR / name).write_bytes(...) 把 bytes 直接写到目标路径

检查一下图文信息是否已经完整，训练集测试集的样本数量是否符合设置。如果存在 missing 情况，直接将 missing 数据从训练集和测试集中删去。

In [ ]:
missing_images = [
    rec["image"] for rec in subset if not (IMAGE_DIR / rec["image"]).exists()
]
print("missing images:", len(missing_images), missing_images[:5])

# 如果有遗失数据，从训练集额和测试集 json 中删去相关记录
if missing_images:
    missing_set = set(missing_images)   # 转成哈希表，下面调用 in 操作更快
    # 从 json 中丢弃 missing 的图文对信息
    train_records = [rec for rec in train_records if rec["image"] not in missing_set]
    val_records = [rec for rec in val_records if rec["image"] not in missing_set]
    # 将修改后的 json 重新写回磁盘
    with open(DATA_ROOT / "chat_train.json", "w", encoding="utf-8") as f:
        json.dump(train_records, f, ensure_ascii=False)
    with open(DATA_ROOT / "chat_val.json", "w", encoding="utf-8") as f:
        json.dump(val_records, f, ensure_ascii=False)

# 打印经过遗失处理后，可用的训练集和测试集样本量，以及文件夹中实际存在的解压后的图片数量
print("usable train:", len(train_records), "val:", len(val_records))
print("extracted images:", len(list(IMAGE_DIR.glob("*.jpg"))))

## 创建 torch.Dataset

使用 torch.Dataset 来规范化自建数据集的接口。Dataset 最重要的功能是提供随机访问接口，即类似于 List 那样**通过方括号指定 index，能够返回一份可以直接给模型前向传播的样本**。先思考一下应该如何构造一个 LLaVA 模型的训练样本。

### LLaVA 训练样本

先思考 label，即提供给模型输入的数据应该如何构建。

我们已经有 `tokenizer` 和 `processor`，通过 `tokenizer` 的 `apply_chat_template`（不 tokenize） 能够将输入的 `human`（指令）进行结构化，而 `processor` 的 `forward` 能够把对文本进行 tokenize 并将图文对完整数据（不像之前图片只是个文件名存在 json 中）在内存中配对，通过字典返回，构成一份完整的样本。

而这里我们多了一份来自 `gpt` 的文本，这是我们期望模型学到的输出。所以应该把这份文本，同样进行 `tokenize` 后连接到由 `human` 转换而来的文本 index 序列后，最后还要追加一个 `<eos>` 标识文本结束。

用 train_records 中的第一个样本做示范处理。

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from transformers import LlavaProcessor

record = train_records[0]

首先提取出 `human` 和 `gpt` 的文字段，并使用 `image` 文件名将真实的图片加载进内存中。

In [ ]:
human = record["conversations"][0]["value"]
answer = record["conversations"][1]["value"]
image = Image.open(IMAGE_DIR / record["image"]).convert("RGB")

human, answer, image

构建格式化的文本，并使用 processor 连接图文。

In [ ]:
# 读取搭建 notebook 中保存的 processor
processor = LlavaProcessor.from_pretrained(MODEL_DIR)

# 格式化信息
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": human},
]
prompt = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
q_inputs = processor(text=prompt, images=image, return_tensors="pt")

q_ids = q_inputs["input_ids"]                # (1, Lq)
pixel_values = q_inputs["pixel_values"][0]   # (3, 336, 336)

prompt, q_ids, pixel_values

将来自 `gpt` 的文本词元化，并与 `<eos>` 一起，concat 到 `q_ids` 后方，组成模型真正的 `input_ids`。

In [ ]:
a_ids = processor.tokenizer(
    answer, add_special_tokens=False, return_tensors="pt"
)["input_ids"]  # [1, La]
eos = torch.tensor([[processor.tokenizer.eos_token_id]], dtype=a_ids.dtype) # [1, 1]

input_ids = torch.cat([q_ids, a_ids, eos], dim=1)[0]    # concat 后统一去掉批量维度

`input_ids` 是完整的序列，也就是给模型输入端的序列。对于输出端，我们需要准备用于计算 loss 的序列。显然我们不希望最终的损失包含来自 `q` 的部分，这里可以**通过交叉熵损失函数的忽略 index 机制实现这一点**。只需要在 labels 中把不希望计算损失的部分标记为 忽略id（记为label_pad_token_id），调用损失函数时传入同一 id，计算结果就不会包含这一部分。

In [ ]:
label_pad_token_id = -100
labels = torch.cat([torch.full_like(q_ids, label_pad_token_id), a_ids, eos], dim=1)[0]

#### 再谈 processor 的细节

记得 processor 的输出是一个 keys 为 "input_ids"、"attention_mask" 和 "pixel_values" 的字典。

实际上 processor 是能够处理批量化的数据的，"attention_mask" 就是用于对齐批量文本长度时，把 `<pad>` 在注意力中屏蔽掉。这里只是做 Dataset，也就是只处理单个样本，所以 "attention_mask" 是一个全 1 的矩阵。

"pixel_values" 是初始图片经过 processor.image_processor 后的图片信息。在我们的例子中，相当于被 CLIP 的图片处理单元处理。同样是处理单个样本，得到的张量形状为 [batch_size=1, channel=3, h=336, w=336]，且其中的数值已经被 CLIP 处理单元内部进行归一化等操作。

同 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中所说，在较新的 transformers 版本中 "input_ids" 会被注入 `num_patches` 个 `<image>`，而现在的版本并不会（把注入过程交给网络的 forward）。对于数据读取只是影响了 "input_ids" 的长度。

### 构建 Dataset

模仿对上面单样本的处理过程，我们可以构建出具有目标功能的 Dataset。

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset

SYSTEM_PROMPT = "You are a helpful assistant."
IMAGE_TOKEN = "<image>"


class LlavaPretrainDataset(Dataset):
    def __init__(self, records_path: Path, image_dir: Path, processor: LlavaProcessor):
        """
        records_path：数据集 json 文件所在路径
        image_dir：数据集解压后的图片所在路径
        """
        with open(records_path, encoding="utf-8") as f:
            self.records = json.load(f)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.image_token_id = self.tokenizer.encode(IMAGE_TOKEN)[0]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        human = record["conversations"][0]["value"]
        # 防御性处理：避免 human 中有不唯一的 <image> 和 gpt 中含有 <image> 的情况
        if human.count(IMAGE_TOKEN) != 1:
            human = IMAGE_TOKEN + "\n" + human.replace(IMAGE_TOKEN, "").strip()
        answer = record["conversations"][1]["value"].replace(IMAGE_TOKEN, "").strip()
        image = Image.open(self.image_dir / record["image"]).convert("RGB")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": human},
        ]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        q_inputs = self.processor(text=prompt, images=image, return_tensors="pt")
        q_ids = q_inputs["input_ids"]                # (1, Lq)
        pixel_values = q_inputs["pixel_values"][0]   # (3, 336, 336)

        a_ids = self.tokenizer(
            answer, add_special_tokens=False, return_tensors="pt"
        )["input_ids"]
        eos = torch.tensor([[self.tokenizer.eos_token_id]], dtype=a_ids.dtype)

        input_ids = torch.cat([q_ids, a_ids, eos], dim=1)[0]
        labels = torch.cat([torch.full_like(q_ids, -100), a_ids, eos], dim=1)[0]

        return {
            "input_ids": input_ids,
            "labels": labels,
            "pixel_values": pixel_values,
            "attention_mask": torch.ones_like(input_ids),
        }

## 创建 Collator

Collator 翻译为 “数据整理器”，可以理解为构建 batch 的"形状对齐器"。自然语言处理中输入的文本长度通常是不同的，为了利用矩阵运算，必须通过 padding 对齐一个 batch 中的所有样本的长度。同样适用特殊 token 进行 padding 能区分出两种不同的 padding 方式：左 `<pad>` 填充 和 右 `<pad>` 填充。使用 右 `<pad>` 填充 更加符合直觉以及更加便捷，但我们选择使用 左 `<pad>` 填充。

### 左 `<pad>` 填充

批内最长样本长度是 `max_len`，这个样本不做 pad。而批内其他样本会在左边补 `<pad>`，真实 prompt + answer + eos 全部右对齐到张量末尾。如：

```markdown
**样本1 (短)**: `[<pad>, <pad>, <pad>, ..., "Provide a brief description of the given image.\n<image>"]`
**样本2 (长)**: `["<image>\nWrite a terse but informative summary of the picture."]`
**样本3 (中)**: `[<pad>, <pad>, ..., "Share a concise interpretation of the image provided.\n<image>"]`
```
注意这些示例是没有添加 gpt 段的，与我们上面的 Dataset 取出的数据不同。

除了对 `input_ids` 进行修改，`attention_mask` 和 `labels` 也需要同步修改。我们不希望模型注意力放到 `<pad>` 词元，同样不希望损失函数中包含 `<pad>` 词元的部分。所以 `attention_mask` 中需要将左填充时补出来的 `pad_len` 个位置填 0，`labels` 中需要将左填充时补出来的 `pad_len` 个位置填 `label_pad_token_id = -100`，**而不是 pad token id**。

在批量推理时，左填充能确保模型每次都从**最后一个真实 token** 的位置开始预测下一个词，而右填充会导致模型从填充符（PAD）开始生成，从而产生错误或空白的输出。为了**和推理/生成链路保持一致**，方便同一套 collator 复用，选用 左 `<pad>` 填充。

In [ ]:
@dataclass
class TrainCollator:
    pad_token_id: int
    label_pad_token_id: int = -100

    def __call__(self, features):
        """
        features: List，元素为 Dataset 中方括号访问 id 取出的数据
        """
        max_len = max(feature["input_ids"].size(0) for feature in features)

        input_ids, labels, attention_mask = [], [], []
        for feature in features:
            length = feature["input_ids"].size(0)
            pad_len = max_len - length

            input_ids.append(
                torch.cat([
                    torch.full((pad_len,), self.pad_token_id, dtype=feature["input_ids"].dtype),
                    feature["input_ids"],
                ])
            )
            labels.append(
                torch.cat([
                    torch.full((pad_len,), self.label_pad_token_id, dtype=feature["labels"].dtype),
                    feature["labels"],
                ])
            )
            attention_mask.append(
                torch.cat([
                    torch.zeros(pad_len, dtype=torch.long),
                    feature["attention_mask"],
                ])
            )

        return {
            "input_ids": torch.stack(input_ids),
            "labels": torch.stack(labels),
            "attention_mask": torch.stack(attention_mask),
            "pixel_values": torch.stack([feature["pixel_values"] for feature in features]),
        }

检查一条样本：prompt 是否正确、`<image>` 是否只有一个、labels 的边界是否正确。

In [ ]:
processor = LlavaProcessor.from_pretrained(MODEL_DIR)

# 使用 Dataset 加载训练数据集和测试数据集
train_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_train.json", IMAGE_DIR, processor)
val_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_val.json", IMAGE_DIR, processor)

sample = train_dataset[0]
print("decoded prompt + answer:")
print(processor.tokenizer.decode(sample["input_ids"], skip_special_tokens=False))
print()
print("input_ids:", tuple(sample["input_ids"].shape))
# 输入中有几个 <image>
print("image tokens in input:", int((sample["input_ids"] == train_dataset.image_token_id).sum()))
# 有多少输入序列被损失函数忽略、没忽略
print("labels == -100:", int((sample["labels"] == -100).sum()))
print("answer labels:", int((sample["labels"] != -100).sum()))
# 输入图片形状
print("pixel_values:", tuple(sample["pixel_values"].shape))

batch = TrainCollator(processor.tokenizer.pad_token_id)([train_dataset[0], train_dataset[1]])
print({key: tuple(value.shape) for key, value in batch.items()})